# Sertifikasi Kompetensi Data Scientist (BNSP)
## Proyek: Pemodelan Prediksi Penyakit Jantung (Dataset: `heart.csv`)

Dokumentasi ini disusun secara terstruktur berdasarkan standar kompetensi **Data Scientist (BNSP)** untuk mendokumentasikan perancangan serta evaluasi pipeline Machine Learning secara *end-to-end*.

## 1. Business Understanding (J.62DMI00.001.1)
Langkah 1. Menentukan Objektif Bisnis

### Permasalahan Bisnis
Penyakit kardiovaskular (CVD) menjadi penyebab utama morbiditas, disabilitas, dan kematian dini secara global. Risiko dan tingkat keparahan kondisi ini dapat ditekan secara signifikan melalui identifikasi dini dan strategi perawatan proaktif. Oleh karena itu, diperlukan sistem yang dapat mengestimasi probabilitas seorang pasien mengalami kejadian klinis kardiovaskular secara akurat.

### Business/Research Objective
Menyelaraskan dengan metodologi pada penelitian *Unified Approach* (Rao et al., 2026), objektif utama proyek ini adalah:
1. Mengestimasi probabilitas atau mengidentifikasi risiko seorang individu mengalami kejadian kardiovaskular secara dini dan proaktif.
2. Membangun pipeline Machine Learning yang andal untuk memprediksi penyakit jantung sebagai alternatif unggul terhadap model risiko konvensional guna meningkatkan akurasi estimasi kondisi kesehatan (*health outcome predictions*).

### Success Metric
- **Akurasi > 85%**
- **Recall > 90%** (Prioritas utama karena meminimalkan *False Negative* sangat krusial dalam diagnosis medis untuk menghindari keterlambatan penanganan klinis)
- **ROC-AUC tinggi** (Kemampuan pemisahan kelas yang optimal)
- Model dapat digunakan sebagai sistem pendukung keputusan klinis (*Clinical Decision Support System*)

### Analisis Risiko & Konsekuensi
- **False Positive (Pasien Sehat diprediksi Sakit)**
  - *Konsekuensi:* Pemeriksaan klinis penunjang tambahan yang tidak diperlukan, peningkatan biaya operasional medis, dan kecemasan pasien.
- **False Negative (Pasien Sakit diprediksi Sehat)**
  - *Konsekuensi:* Keterlambatan intervensi medis proaktif yang berpotensi fatal bagi keselamatan pasien.
  
### Manfaat Proyek
- Mendukung deteksi dini penyakit jantung secara akurat.
- Membantu dokter dalam screening awal risiko kardiovaskular.
- Efisiensi biaya dan waktu penanganan klinis melalui klasifikasi berbasis risiko.

## 2. Technical Understanding (Kompetensi J.62DMI00.002.1)
Langkah 2. Menentukan Tujuan Teknis Data Science

### Spesifikasi Teknis
- **Tugas Pemodelan:** Klasifikasi Biner (Target: 0 = Sehat, 1 = Penyakit Jantung)
- **Input:** 13 Fitur Medis (Kategorikal dan Numerik)
- **Output:** Kelas Target Prediktif
- **Teknologi:** Python, Pandas, Numpy, Matplotlib, Seaborn, Scikit-learn

In [ ]:
# Pemuatan pustaka dasar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pustaka preprocessing dan seleksi model
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Pustaka algoritma pemodelan
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Pustaka metrik evaluasi
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

# Konfigurasi estetika visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Seluruh pustaka berhasil dimuat.")

## 3. Data Understanding (Kompetensi J.62DMI00.005.1)
Langkah 3. Menelaah Data

### Eksplorasi Data Awal
Proses pemuatan dataset `heart.csv` untuk analisis struktur, dimensi, dan tipe data awal.

In [ ]:
# Pemuatan dataset
df = pd.read_csv('heart.csv')

# Output dimensi dataset
print(f"Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

In [ ]:
# Penelaahan metadata dan tipe data kolom
df.info()

In [ ]:
# Ringkasan statistik deskriptif data numerik
df.describe().T

### Identifikasi Variabel Medis
Berdasarkan karakteristik klinis, variabel dikelompokkan sebagai berikut:
- **Fitur Numerik:** `age`, `trestbps` (resting blood pressure), `chol` (cholesterol), `thalach` (max heart rate), `oldpeak` (ST depression)
- **Fitur Kategorikal:** `sex`, `cp` (chest pain type), `fbs` (fasting blood sugar), `restecg` (resting ECG), `exang` (exercise angina), `slope` (ST slope), `ca` (colored vessels), `thal` (thalassemia type)
- **Target:** `target` (0 = Sehat, 1 = Sakit Jantung)

In [ ]:
# Identifikasi data hilang (missing values)
print("=== Missing Values ===")
print(df.isnull().sum())

# Identifikasi duplikasi data
print("\n=== Duplikasi Baris ===")
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

In [ ]:
# Visualisasi distribusi kelas target
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='target', palette='Set2')
plt.title('Distribusi Kelas Target (0=Sehat, 1=Sakit)', fontweight='bold')
plt.xlabel('Target')
plt.ylabel('Jumlah Pasien')
plt.show()

print(df['target'].value_counts(normalize=True))

In [ ]:
# Analisis distribusi fitur numerik
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='skyblue')
    axes[idx].set_title(f'Distribusi Fitur {col}', fontweight='bold')

# Eliminasi subplot sisa
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi boxplot untuk deteksi pencilan (outliers)
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for idx, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[idx], color='lightcoral')
    axes[idx].set_title(f'Outliers pada {col}', fontweight='bold')

plt.tight_layout()
plt.show()

### Analisis Pencilan (Outliers)
Berdasarkan visualisasi boxplot pada lima fitur numerik, teridentifikasi adanya nilai-nilai pencilan sebagai berikut:
1. **Identifikasi Fitur Terpengaruh:**
   - **`trestbps` (Tekanan Darah Istirahat):** Terdapat beberapa pencilan di atas batas atas (khususnya nilai > 170 mmHg).
   - **`chol` (Kolesterol Serum):** Terdeteksi beberapa pencilan bernilai tinggi, dengan nilai ekstrem mencapai > 350 mg/dl.
   - **`thalach` (Detak Jantung Maksimum):** Ditemukan satu pencilan di bawah batas bawah (< 80 bpm).
   - **`oldpeak` (Depresi ST):** Terdeteksi beberapa pencilan di atas batas normal (> 4.0).

2. **Justifikasi Penanganan Pencilan:**
   - Secara klinis, nilai-nilai ekstrem tersebut bukan disebabkan oleh kesalahan input data, melainkan merepresentasikan kondisi patologis akut pasien penyakit jantung.
   - Berdasarkan hasil eksperimen (mengacu pada Paper Pendukung A), penghapusan pencilan klinis menggunakan algoritma deteksi outlier seperti Isolation Forest terbukti menurunkan metrik *Recall* model secara signifikan (misalnya, *Recall* XGBoost turun dari 83% menjadi 73%). Hal ini disebabkan karena model kehilangan informasi vital mengenai indikator klinis penyakit kritis. Oleh karena itu, seluruh pencilan dipertahankan untuk menjaga sensitivitas deteksi medis.

In [ ]:
# Visualisasi matriks korelasi antar fitur
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, square=True)
plt.title('Matriks Korelasi Fitur', fontweight='bold', fontsize=14)
plt.show()

### Formulasi Hipotesis Medis
1. **Faktor Usia (`age`):** Pertambahan usia memiliki korelasi positif dengan peningkatan risiko penyakit jantung.
2. **Karakter Nyeri Dada (`cp`):** Tipe nyeri dada non-anginal atau *asymptomatic* memiliki relevansi tinggi terhadap diagnosis positif.
3. **Detak Jantung Maksimum (`thalach`):** Nilai detak jantung maksimum yang lebih rendah berkaitan dengan kondisi patologis.
4. **Depresi ST (`oldpeak`):** Nilai depresi ST yang tinggi setelah aktivitas fisik berkorelasi kuat dengan penyakit jantung.

Hipotesis ini dievaluasi lebih lanjut melalui koefisien model dan tingkat kepentingan fitur (*feature importance*).

## 4. Data Validation (Kompetensi J.62DMI00.006.1)
Langkah 4. Memvalidasi Data

### Evaluasi Kualitas Data
- **Kelengkapan Data:** Tidak ditemukan nilai kosong (0 missing values) di seluruh kolom.
- **Konsistensi Data:** Terdeteksi 723 baris duplikat dari total 1.025 baris. Duplikasi ini terjadi karena replikasi data historis. Guna menghindari bias evaluasi (*data leakage*) pada pembagian data latih-uji, baris duplikat wajib dibersihkan.
- **Validitas Klinis:** Terdapat nilai ekstrem pada variabel `chol` (> 400 mg/dl) dan `trestbps` (> 180 mmHg) yang merepresentasikan variasi biologis riil pasien patologis.

## 5. Data Selection (Kompetensi J.62DMI00.007.1)

### Penentuan Objek Data
Seluruh 13 fitur medis dipertahankan karena memiliki relevansi klinis yang signifikan berdasarkan literatur medis. Variabel target dipisahkan sebagai label prediksi.

In [ ]:
# Seleksi fitur prediktor (X) dan label target (y) dari dataset awal
X_selected = df.drop(columns=['target'])
y_selected = df['target']

print("Fitur prediktor terpilih:", list(X_selected.columns))
print("Target terpilih: ['target']")

## 6. Data Cleaning (Kompetensi J.62DMI00.008.1)

### Protokol Pembersihan Data
- **Penanganan Duplikasi:** Baris duplikat dibersihkan menggunakan fungsi `.drop_duplicates()`, menyisakan 302 baris data unik.
- **Penanganan Missing Values:** Hasil telaah menunjukkan tidak adanya data yang hilang, sehingga prosedur imputasi tidak diperlukan.
- **Penanganan Outlier:** Pencilan nilai klinis dipertahankan karena merupakan indikasi kondisi patologis akut yang valid, bukan kesalahan input data.

In [ ]:
# Pembersihan data duplikat pada dataset
df_clean = df.drop_duplicates().reset_index(drop=True)
X_clean = df_clean.drop(columns=['target'])
y_clean = df_clean['target']

print(f"Dimensi data sebelum pembersihan: {df.shape[0]} baris")
print(f"Dimensi data setelah pembersihan duplikat: {df_clean.shape[0]} baris")

## 7. Data Construction (Kompetensi J.62DMI00.009.1)

### Konstruksi dan Transformasi Fitur
- **Encoding Kategori:** Transformasi *One-Hot Encoding* diterapkan pada fitur kategorikal multikelas (`cp`, `restecg`, `slope`, `ca`, `thal`) untuk menghindari asumsi urutan (*ordinal bias*).
- **Standardisasi Fitur:** Penerapan `StandardScaler` pada fitur numerik kontinu guna menyamakan skala kontribusi fitur, terutama untuk algoritma linier (Logistic Regression) dan berbasis jarak.
- **Integrasi Pipeline:** Seluruh alur preprocessing dibungkus menggunakan `ColumnTransformer` dan `Pipeline` scikit-learn untuk memastikan ketahanan model terhadap kebocoran data.
- **Partisi Data:** Pembagian data dilakukan secara acak terstratifikasi (`stratify=y`) dengan rasio 80% data latih dan 20% data uji.

In [ ]:
# ── 1. Partisi Data Latih & Uji (80:20 Stratified) ─────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.20, random_state=42, stratify=y_clean
)
print(f"Training set: {X_train.shape[0]} sampel")
print(f"Test set    : {X_test.shape[0]} sampel")

# ── 2. Konstruksi Preprocessing Pipeline ──────────────────────────
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['sex', 'fbs', 'exang', 'cp', 'restecg', 'slope', 'ca', 'thal']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)

print("\nPipeline Preprocessing berhasil dikonstruksi!")

## 8. Modeling Scenario (Kompetensi J.62DMI00.012.1)

### Desain Skenario Pemodelan
Evaluasi komparatif dilakukan terhadap 4 algoritma klasifikasi:
1. **Logistic Regression:** Model linier sebagai baseline dengan interpretabilitas matematis yang tinggi.
2. **Decision Tree Classifier:** Model berbasis pohon keputusan sebagai pembanding non-linier awal.
3. **Random Forest Classifier:** Algoritma ensemble berbasis bagging untuk mengurangi varians.
4. **Gradient Boosting Classifier:** Algoritma ensemble berbasis boosting untuk meminimalkan bias secara bertahap.

### Protokol Evaluasi
- **Validasi Silang:** Penerapan *Stratified 5-Fold Cross-Validation* pada data latih untuk optimasi hyperparameter.
- **Metrik Utama:** Skor **Recall** ditetapkan sebagai metrik optimasi utama untuk menekan rasio *False Negative* klinis. Metrik akurasi, F1-score, dan ROC-AUC digunakan sebagai metrik komplementer.

## 9. Modeling & Hyperparameter Tuning (Kompetensi J.62DMI00.013.1)

### Pelatihan Model dan Optimasi Hyperparameter
Pencarian parameter optimal dilakukan melalui eksperimen kombinasi hyperparameter menggunakan metode `GridSearchCV` yang diintegrasikan dalam pipeline validasi silang.

In [ ]:
# Konstruksi pipeline model terintegrasi
pipelines = {
    'Logistic Regression': Pipeline([('prep', preprocessor), ('clf', LogisticRegression(random_state=42))]),
    'Decision Tree': Pipeline([('prep', preprocessor), ('clf', DecisionTreeClassifier(random_state=42))]),
    'Random Forest': Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(random_state=42))]),
    'Gradient Boosting': Pipeline([('prep', preprocessor), ('clf', GradientBoostingClassifier(random_state=42))])
}

# Definisikan ruang hyperparameter pencarian
param_grids = {
    'Logistic Regression': {
        'clf__C': [0.01, 0.1, 1.0, 10.0],
        'clf__penalty': ['l2']
    },
    'Decision Tree': {
        'clf__max_depth': [3, 5, 7, None],
        'clf__min_samples_split': [2, 5, 10]
    },
    'Random Forest': {
        'clf__n_estimators': [50, 100, 200],
        'clf__max_depth': [3, 5, 7, None],
        'clf__min_samples_split': [2, 5]
    },
    'Gradient Boosting': {
        'clf__n_estimators': [50, 100, 150],
        'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'clf__max_depth': [3, 4, 5]
    }
}

best_models = {}
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Proses Penyetelan Hyperparameter Menggunakan GridSearchCV (5-Fold CV) ===")
for name, pipeline in pipelines.items():
    grid = GridSearchCV(
        pipeline, 
        param_grids[name], 
        cv=cv_strategy, 
        scoring='recall',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    print(f"Parameter Terbaik untuk {name}: {grid.best_params_}")
    print(f"Skor Recall Terbaik (CV)      : {grid.best_score_:.4f}\n")

## 10. Evaluation (Kompetensi J.62DMI00.014.1)

### Evaluasi Kinerja pada Data Uji (Test Set)
Pengujian performa akhir model dilakukan menggunakan subset data uji (*test set*) untuk menilai kemampuan generalisasi pada data baru.

In [ ]:
results = []
y_preds = {}
y_probs = {}

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_preds[name] = y_pred
    y_probs[name] = y_prob
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

df_eval = pd.DataFrame(results).set_index('Model')
print("=== Tabel Evaluasi Kinerja Model Akhir pada Test Set ===")
df_eval.round(4)

In [ ]:
# Visualisasi Matriks Kebingungan (Confusion Matrix)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, _) in enumerate(best_models.items()):
    cm = confusion_matrix(y_test, y_preds[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Sehat', 'Sakit'], yticklabels=['Sehat', 'Sakit'])
    axes[idx].set_title(f'Confusion Matrix — {name}', fontweight='bold')
    axes[idx].set_xlabel('Prediksi')
    axes[idx].set_ylabel('Aktual')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi Kurva ROC-AUC Gabungan
plt.figure(figsize=(10, 8))
ax = plt.gca()

for name, model in best_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)

plt.plot([0, 1], [0, 1], 'k--', label='Chance Level (AUC = 0.50)')
plt.title('Kurva ROC-AUC Model Perbandingan', fontweight='bold', fontsize=14)
plt.legend(loc='lower right')
plt.show()

### Seleksi Model Terbaik
Berdasarkan hasil pengujian pada data uji:
- Model **Logistic Regression** dan **Gradient Boosting** menunjukkan kinerja klasifikasi yang stabil dengan capaian Recall yang optimal.
- Model dengan nilai **Recall tertinggi** dipilih sebagai kandidat utama untuk implementasi produksi guna menjamin keselamatan diagnosis pasien.

## 11. Review & Recommendation (Kompetensi J.62DMI00.015.1)

### Kelayakan Model
- Berdasarkan kriteria performa teknis, model yang dibangun dinilai **layak** untuk diimplementasikan sebagai sistem pendukung keputusan skrining awal klinis karena memenuhi ambang batas keakuratan dan sensitivitas (Recall) yang diharapkan.

### Keunggulan Model
- **Generalisasi Optimal:** Penerapan arsitektur pipeline terisolasi menjamin proses penskalaan (*scaling*) dan penyandian (*encoding*) bebas dari risiko kebocoran data (*data leakage*).
- **Kapatuhan Protokol Data:** Pembersihan baris duplikat yang konsisten di awal alur kerja menjaga keandalan evaluasi statistik model.

### Keterbatasan Model
- **Representasi Demografis:** Dataset bersumber dari data historis internasional (UCI) yang belum tentu merepresentasikan karakteristik fisiologis spesifik populasi pasien lokal di Indonesia.
- **Kuantitas Data:** Proses eliminasi data duplikat mereduksi ukuran dataset menjadi 302 baris unik, yang berpotensi memengaruhi stabilitas estimasi performa.

### Rekomendasi Pengembangan
1. **Validasi Eksternal:** Melakukan pengujian performa model menggunakan dataset klinis lokal dari rumah sakit mitra di Indonesia.
2. **Eksplorasi Model Lanjutan:** Menguji algoritma klasifikasi ensemble tingkat lanjut (seperti XGBoost atau LightGBM) dengan tuning parameter yang lebih komprehensif.
3. **Prototipe Sistem Aplikasi:** Membangun antarmuka pengguna berbasis web sederhana (menggunakan Streamlit atau Flask) untuk memudahkan praktisi medis menginput parameter klinis pasien dan memperoleh hasil prediksi secara real-time.

--- 

### Bagan Alur Kerja Visual (Pipeline CRISP-DM)

```
Business Understanding
        │
        ▼
Technical Understanding
        │
        ▼
Load Dataset (heart.csv)
        │
        ▼
EDA (Info, Missing, Duplicate, Outlier, Correlation, Visualisasi)
        │
        ▼
Data Validation (Evaluasi Kualitas & Identifikasi Duplikasi)
        │
        ▼
Data Cleaning (Pembersihan Duplikat)
        │
        ▼
Feature Selection (Menentukan X & y)
        │
        ▼
Feature Engineering (Encoding, Scaling, Train-Test Split 80:20)
        │
        ▼
Modeling Scenario (Logistic Reg, Decision Tree, Random Forest, Gradient Boosting)
        │
        ▼
Hyperparameter Tuning (GridSearchCV 5-Fold CV)
        │
        ▼
Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix)
        │
        ▼
Review & Recommendation
```